## Decision Tree Classifier with Optuna 
### 1. Goal of the Notebook

In this notebook, we build a Decision Tree Classifier to solve a classification problem and use Optuna to automatically find the best hyperparameters for the model.
👉 Objective:

Train a Decision Tree model
Optimize its performance using hyperparameter tuning


### 2. Loading and Understanding the Data
The notebook starts by:

Loading the dataset using pandas
Exploring:

Dataset shape
Feature types
Missing values
Class distribution



👉 Purpose:
To understand the data and identify preprocessing needs before modeling.

###3. Data Preprocessing
Before training, the notebook processes the data:
a. Handling Missing Values

Filling or removing missing values

b. Encoding Categorical Variables

Converting categories into numerical format

c. Feature Scaling (if included)

Not strictly required for Decision Trees, but sometimes applied

👉 Important note:
Decision Trees are not sensitive to feature scaling, unlike Logistic Regression.

###4. Splitting the Dataset
The dataset is split into:

Training set
Test set

Pythontrain_test_split(X, y, test_size=0.2)``Show more lines
👉 Purpose:
To evaluate model generalization on unseen data.

###5. Decision Tree Model (Baseline)
Initially, the notebook may train a basic Decision Tree:
Pythonfrom sklearn.tree import DecisionTreeClassifiermodel = DecisionTreeClassifier()model.fit(X_train, y_train)Show more lines
What a Decision Tree does:

Splits the data into branches based on feature values
Creates rules like:

“If feature A < value → go left”
“Else → go right”


Continues splitting until:

a stopping condition is met
or leaves are pure



👉 Result:
A tree structure that makes decisions step-by-step.

###6. Why Hyperparameter Tuning Is Needed
Decision Trees can easily:

Overfit (too deep → memorizes training data)
Underfit (too shallow → misses patterns)

Key parameters that control this behavior:

max_depth → maximum depth of the tree
min_samples_split → minimum samples to split a node
min_samples_leaf → minimum samples in a leaf
criterion → split quality measure (gini or entropy)

👉 Purpose:
Find the best combination of these parameters for optimal performance.


In [0]:
%run ../utils/upload_data

In [0]:
# Prepare the dataset config
files = ["diabetes.csv"]
base_volume = "/Volumes/mlpractice/source/mlmodel/diabetes_data"
url = "https://raw.githubusercontent.com/kuljotSB/DatabricksUdemyCourse/refs/heads/main/MachineLearningModel"

In [0]:
# Read Dataset and create Spark DataFrame
base_volume = "/Volumes/dbx_apps_poc/mlpractice/volumes/mlmodel/diabetes_data"

df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{base_volume}/diabetes.csv")
display(df.limit(5))



In [0]:
#split train test data 
train_df, test_df = df.randomSplit([0.7, 0.3])

In [0]:
# Hyperparamere tuning
import mlflow
from mlflow.models import infer_signature

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, MinMaxScaler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

def decitionTree_model_experment(trial, train_df, test_df):
  max_depth = trial.suggest_int("max_depth", 1, 10)
  max_bins = trial.suggest_categorical("max_bins", [10, 20, 30])
  
  with mlflow.start_run(nested=True):
      # Train a model using the provided hyperparameter value
      numericalFeatures = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]
      assembler = VectorAssembler(inputCols=numericalFeatures, outputCol="numericFeatures") # Numerical verctorized data
      scaler = MinMaxScaler(inputCol=assembler.getOutputCol(), outputCol="normalizedFeatures")
      featureVector = VectorAssembler(inputCols=["normalizedFeatures"], outputCol="features")
      dtAlgo = DecisionTreeClassifier(labelCol="Outcome", featuresCol="features", maxDepth=max_depth, maxBins=max_bins)

      pipeline = Pipeline(stages=[assembler, scaler, featureVector, dtAlgo]) # Pipeline
      model = pipeline.fit(train_df)

      # Evaluate the model
      predictions = model.transform(test_df)

      eval = MulticlassClassificationEvaluator(labelCol="Outcome", predictionCol='prediction', metricName= 'accuracy')
      accuracy_score = eval.evaluate(predictions)

      # Log parameters and metrics
      mlflow.log_param('MaxDepth', max_depth)
      mlflow.log_param('MaxBins', max_bins)
      mlflow.log_metric('accuracy', accuracy_score)


      # Infer and log model signiture
      signature = infer_signature(train_df.select(numericalFeatures).toPandas(), predictions.select('prediction').toPandas())
      mlflow.spark.log_model(model, "model", signature=signature, dfs_tmpdir='/Volumes/mlpractice/source/mlmodel/ml_lab/tmp/')
    
  return accuracy_score 


    


In [0]:
# Defining the Search Space for hyperparameters, and log each hyperparamer run using a Trials() object

import optuna
#from optuna.integration.mlflow import MlflowCallback

# Create the optuna study to maximize accuracy
with mlflow.start_run(run_name="DecisionTreeClassifier_Optuna"):
    study = optuna.create_study(direction="maximize", study_name='diabetes_classification')

    # run optimization
    study.optimize(lambda trial: decitionTree_model_experment(trial, train_df, test_df), n_trials=3)

    # Get best parametes 
    print("Best param vales: ", study.best_params)
    print("Best accuracy: ", study.best_value)

    # Log best params to parent run
    mlflow.log_params(study.best_params)
    mlflow.log_metric('best_accuracy', study.best_value )

    # display all trails
    print('\n All trials:')
    for trial in study.trials:
        print(f'\nTrail -> number:  {trial.number}, params: {trial.params}, accuracy: {trial.value}')
